In [6]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
import matplotlib.pyplot as plt

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

# ---- Model Definition ----
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2),
    nn.Softmax(dim=1)
)
model = model.to(device)

Running on: cpu


In [8]:
model.load_state_dict(torch.load(
    r"D:\AI Mini project (AI vs Real image)\resnet18_cifake_classifier.pth",
    map_location=device
))
model.eval()
print("✅ Model loaded successfully!")

# ---- Transform ----
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

✅ Model loaded successfully!


In [9]:
train_dir = r"D:\AI Mini project (AI vs Real image)\archive\train"
test_dir  = r"D:\AI Mini project (AI vs Real image)\archive\test"

# ---- Load Training Dataset ----
full_train_data = datasets.ImageFolder(root=train_dir, transform=transform)

# ---- AUTO SPLIT: 90% Train, 10% Validation ----
val_size = int(0.1 * len(full_train_data))
train_size = len(full_train_data) - val_size

train_data, val_data = random_split(full_train_data, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)

# ---- Load Test Dataset ----
test_data = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
class_names = list(test_data.class_to_idx.keys())

criterion = nn.CrossEntropyLoss()

In [12]:
def evaluate(loader):
    model.eval()
    total_loss = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            total_loss += loss.item()

            preds = out.argmax(1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    return total_loss / len(loader), acc

In [13]:
train_loss, train_acc = evaluate(train_loader)
val_loss, val_acc = evaluate(val_loader)

print("\n====== TRAIN vs VALIDATION METRICS ======")
print(f"📘 Training Accuracy:    {train_acc*100:.2f}%")
print(f"📗 Validation Accuracy:  {val_acc*100:.2f}%")
print(f"📘 Training Loss:        {train_loss:.4f}")
print(f"📗 Validation Loss:      {val_loss:.4f}")



====== TRAIN vs VALIDATION METRICS ======
📘 Training Accuracy:    90.35%
📗 Validation Accuracy:  90.22%
📘 Training Loss:        0.4051
📗 Validation Loss:      0.4065
